[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/00_p07_autograd.ipynb)

# 🔵 Prerequisite 7: Autograd Essentials

Every training-related problem (optimizers, gradient clipping, gradient accumulation, BatchNorm's running stats, DPO/PPO/GRPO losses) depends on knowing **what autograd tracks**, **where gradients end up**, and **how to stop them**.

The classic bugs are silent: gradients that accumulate across steps, a parameter update that gets recorded in the graph, or a gradient leaking into something that should be constant (a target, an old policy, an advantage).

### What you'll learn
1. `requires_grad`, leaf tensors, and where `.grad` appears
2. Gradients **accumulate**: zero them between steps
3. `torch.no_grad()` for updates and initialization
4. `detach()` as a stop-gradient
5. `torch.autograd.grad` for a gradient without touching `.grad`
6. Inspecting gradients: `inference_mode`, tensor hooks, `gradcheck`, anomaly detection
7. Many ways to do it: stopping gradients, and their differences

### Exercises
| # | Function | Shows up in |
|---|----------|-------------|
| 1 | `grad_of(f, x)` | checking gradients by hand, #40 Linear Regression |
| 2 | `sgd_step_(params, lr)` | #29 Adam, #40 Linear Regression |
| 3 | `stop_gradient_mse(pred, target)` | #37 DPO, #38 GRPO, #39 PPO, #41 OPD |
| 4 | `accumulate_grads(w, batches)` | #31 Gradient Accumulation |
| 5 | `straight_through_round(x)` | quantization-aware training (#36) |

### Rules
- **No Python loops** over tensor elements. (Looping over a *list* of parameters or micro-batches is fine.)
- Do **not** use `torch.optim`.

### 📐 Notation

| Symbol | Meaning |
|:---:|---|
| `D` | number of parameters in `w` |
| `n` | examples per micro-batch |

### How to use this notebook
1. **Run the lesson cells** (sections 1, 2, …) from top to bottom. They are demos: change them and re-run.
2. **Do the exercises in order.** Each one has its own `✏️ YOUR IMPLEMENTATION HERE` cell, followed by a debug cell for checking shapes before you submit.
3. **Submit at any time** with the `check("p07_autograd")` cell at the bottom. It grades every exercise at once, and each test is named after its exercise, so a partial score shows which ones are left.
4. **Stuck?** `hint("p07_autograd")` gives one hint per exercise; `00_p07_autograd_solution.ipynb` has the reference code.
5. **Already comfortable with this skill?** Prerequisites are optional. Skip straight to the problems listed in **🔓 What this unlocks** at the bottom.

📖 **Reference solution:** [`00_p07_autograd_solution.ipynb`](00_p07_autograd_solution.ipynb) (JupyterLab) · [`solutions/00_p07_autograd_solution.ipynb`](../solutions/00_p07_autograd_solution.ipynb) (GitHub / VS Code). Try it yourself first!

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

## 1. Leaf tensors and `.grad`

A tensor you create with `requires_grad=True` is a **leaf**. Every operation on it records a node in a graph. Calling `.backward()` on a **scalar** result fills `.grad` on the **leaves**.
Intermediate (non-leaf) tensors don't keep `.grad` unless you ask with `.retain_grad()`.

In [ ]:
w = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = (w ** 2)              # intermediate
loss = y.sum()
loss.backward()
print(w.is_leaf, y.is_leaf)        # True False
print(w.grad)                      # tensor([2., 4., 6.]) = d(sum w^2)/dw
print(y.grad)                      # None (with a warning): non-leaf

## 2. ⚠️ Gradients accumulate

`.backward()` **adds** to `.grad`. It doesn't overwrite it. That's what makes gradient accumulation (#31) work, and it's also why every training loop calls `zero_grad()` before each step.

In [ ]:
w = torch.tensor([1.0, 2.0], requires_grad=True)
(w * 3).sum().backward()
print(w.grad)              # tensor([3., 3.])
(w * 3).sum().backward()
print(w.grad)              # tensor([6., 6.]): added, not replaced
w.grad = None              # or w.grad.zero_()

## 3. `torch.no_grad()` for updates

Updating a parameter in place (`w -= lr * w.grad`) must **not** be recorded in the graph. PyTorch refuses in-place ops on a leaf that requires grad unless you wrap them in `torch.no_grad()`.
Weight initialization (`weight.normal_(...)` in #20) and running-stat updates (#07) use the same pattern.

In [ ]:
w = torch.tensor([1.0, 2.0], requires_grad=True)
(w ** 2).sum().backward()
try:
    w -= 0.1 * w.grad
except RuntimeError as e:
    print("RuntimeError:", str(e)[:70])
with torch.no_grad():
    w -= 0.1 * w.grad
print(w, w.is_leaf, w.requires_grad)   # still a trainable leaf

## 4. `detach()` as a stop-gradient

`x.detach()` returns the same values, cut off from the graph. Anything computed from it is treated as a **constant** during backward.
Use it whenever a quantity must not be trained through: regression targets, PPO's old log-probs, GRPO's advantages, a teacher's logits, BatchNorm's running stats.

A useful trick: `x + (f(x) - x).detach()` has the *value* of `f(x)` but the *gradient* of `x`. This is the **straight-through estimator**.

In [ ]:
x = torch.tensor([2.0], requires_grad=True)
a = (x * x).sum()                  # d/dx = 2x = 4
b = (x * x.detach()).sum()         # second factor is a constant: d/dx = x = 2
print(torch.autograd.grad(a, x)[0], torch.autograd.grad(b, x)[0])   # tensor([4.]) tensor([2.])

## 5. `torch.autograd.grad`

`torch.autograd.grad(output, inputs)` returns the gradients as a tuple **without** writing to `.grad`. It's handy for checking a gradient you derived by hand, or for gradients of gradients.

In [ ]:
x = torch.tensor([1.0, -2.0], requires_grad=True)
(g,) = torch.autograd.grad((x ** 3).sum(), x)
print(g, x.grad)          # tensor([3., 12.]) None: .grad untouched

## 6. Inspecting gradients

- **`torch.inference_mode()`** is a faster, stricter `no_grad()` for pure inference: tensors created inside can **never** be used in autograd later. Use `no_grad()` for things like parameter updates, `inference_mode()` for serving and evaluation.
- **`t.register_hook(fn)`** calls `fn(grad)` when the gradient of `t` is computed. Use it to print, log, or even modify a gradient in flight (it works on intermediate tensors too).
- **`torch.autograd.gradcheck(fn, inputs)`** compares your gradients against finite differences. Use `float64` inputs, or the check is too noisy.
- **`torch.autograd.detect_anomaly()`** makes backward raise an error at the **first** operation that produced `NaN`, with a traceback to the forward line that created it.

In [ ]:
with torch.inference_mode():
    y = torch.ones(2) * 3
print(y.is_inference())                        # True: can't be fed into autograd later

x = torch.tensor([1.0, 2.0], requires_grad=True)
h = x * 3
h.register_hook(lambda g: print("grad reaching h:", g))
(h ** 2).sum().backward()                      # prints the gradient flowing into h
print(x.grad)

f = lambda t: (t.sin() * t).sum()
print(torch.autograd.gradcheck(f, (torch.randn(4, dtype=torch.float64, requires_grad=True),)))   # True

z = torch.tensor([0.0], requires_grad=True)
try:
    with torch.autograd.detect_anomaly():
        torch.sqrt(z - 1).sum().backward()      # sqrt(-1) = nan
except RuntimeError as e:
    print("anomaly:", str(e)[:70])

## 7. Many ways to do it: stopping gradients

| Form | What it does | Use for |
|------|--------------|---------|
| `x.detach()` | same values, cut out of the graph; **shares memory** with `x` | constants inside a loss (targets, old log-probs) |
| `with torch.no_grad():` | nothing inside is recorded | parameter updates, running stats, evaluation |
| `with torch.inference_mode():` | like `no_grad`, faster, results can never enter autograd | pure inference |
| `p.requires_grad_(False)` | stops computing gradients **for a leaf** from now on | freezing parameters (P8) |
| `x.detach().clone()` | a cut-off **copy** | when you'll modify the copy in place |

`x.data` also exists in old code. Avoid it: autograd doesn't track changes made through it, so it can silently corrupt gradients.

In [ ]:
x = torch.tensor([1.0, 2.0], requires_grad=True)
d = x.detach()
print(d.data_ptr() == x.data_ptr())          # True: shares memory
d2 = x.detach().clone()
print(d2.data_ptr() == x.data_ptr())         # False: an independent copy
with torch.no_grad():
    y = x * 2
print(y.requires_grad, (x * 2).requires_grad)  # False True

---
# ✏️ Exercises

Implement the five functions below. Each one has a debug cell you can use before you submit.

### Exercise 1: `grad_of(f, x)`

Return the gradient of the scalar `f(x)` with respect to `x`.

- `f`: a function from a tensor to a **scalar** tensor
- `x`: a tensor, which may or may not have `requires_grad`
- returns: a tensor with the shape of `x`
- must **not** change `x` or `x.grad`

*Comparing autograd with a hand-derived gradient is how you check manual-gradient code like #40's gradient descent.*

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def grad_of(f, x: torch.Tensor) -> torch.Tensor:
    pass  # Replace this

In [ ]:
x = torch.tensor([1.0, -2.0, 3.0])
print(grad_of(lambda t: (t ** 2).sum(), x))   # expect tensor([ 2., -4.,  6.])

### Exercise 2: `sgd_step_(params, lr)`

One step of plain gradient descent, then clear the gradients, as `optimizer.step()` followed by `optimizer.zero_grad()` would.

- `params`: a list of leaf tensors with `requires_grad=True`; some may have `.grad is None` (skip those)
- `lr`: float
- update each parameter **in place**: `p ← p - lr * p.grad`
- afterwards every parameter must still be a leaf with `requires_grad=True`, and its `.grad` must be `None` or all zeros
- returns: nothing

*#29 Adam is the same loop with a smarter update.*

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def sgd_step_(params: list[torch.Tensor], lr: float) -> None:
    pass  # Replace this

In [ ]:
w = torch.tensor([1.0, 2.0], requires_grad=True)
(w ** 2).sum().backward()          # grad = 2w = [2, 4]
sgd_step_([w], lr=0.1)
print(w, w.grad)                   # expect tensor([0.8000, 1.6000], requires_grad=True) None

### Exercise 3: `stop_gradient_mse(pred, target)`

Mean squared error where gradients flow into `pred` **only**. `target` is treated as a constant, even if it requires grad.

- `pred`, `target`: same shape
- returns: a scalar tensor, `mean((pred - target) ** 2)`

*PPO, GRPO, DPO and OPD all detach something: the old log-probs, the advantages, the reference model, the teacher.*

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def stop_gradient_mse(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    pass  # Replace this

In [ ]:
pred = torch.tensor([1.0, 2.0], requires_grad=True)
target = torch.tensor([0.0, 0.0], requires_grad=True)
stop_gradient_mse(pred, target).backward()
print(pred.grad, target.grad)      # expect tensor([1., 2.]) None

### Exercise 4: `accumulate_grads(w, batches)`

Gradient accumulation: compute the gradient of the **average** loss over several equally sized micro-batches, calling `.backward()` once per micro-batch.

- `w`: leaf tensor `(D,)` with `requires_grad=True`; it may already hold an old `.grad`
- `batches`: a list of `(x, y)` pairs, `x`: `(n, D)`, `y`: `(n,)`
- loss of one micro-batch: `mean((x @ w - y) ** 2)`
- returns: `w.grad`, which must equal the gradient of the mean of the micro-batch losses

*Two things to get right: clear the old gradient first, and scale each micro-batch loss. This is #31.*

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def accumulate_grads(w: torch.Tensor, batches: list[tuple[torch.Tensor, torch.Tensor]]) -> torch.Tensor:
    pass  # Replace this

In [ ]:
torch.manual_seed(0)
w = torch.randn(3, requires_grad=True)
X, Y = torch.randn(8, 3), torch.randn(8)
batches = [(X[:4], Y[:4]), (X[4:], Y[4:])]
g = accumulate_grads(w, batches).clone()
full = grad_of(lambda t: ((X @ t - Y) ** 2).mean(), w)
print(g, full)          # expect the same values (uses your grad_of from Exercise 1)

### Exercise 5: `straight_through_round(x)`

Round to the nearest integer in the **forward** pass, but let the gradient pass through **unchanged** in the backward pass, as if rounding were the identity.

- `x`: float tensor
- returns: same shape, values equal to `torch.round(x)`, and `d out / d x = 1` everywhere

*`torch.round` alone has zero gradient almost everywhere, so nothing could learn through it. Quantization-aware training (#36) uses this trick.*

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def straight_through_round(x: torch.Tensor) -> torch.Tensor:
    pass  # Replace this

In [ ]:
x = torch.tensor([0.2, 1.7, -2.6], requires_grad=True)
out = straight_through_round(x)
out.sum().backward()
print(out, x.grad)      # expect tensor([ 0.,  2., -3.]) tensor([1., 1., 1.])

<!-- Generated from scripts/prereqs/content/p07_autograd.py by `make prereqs`. Edit that file instead of this notebook; manual edits are overwritten. -->
## ✅ Submit

Run the judge on **all** exercises in this notebook. Each test is labelled with the exercise it checks.

In [ ]:
from torch_judge import check, hint
check("p07_autograd")

In [ ]:
# 💡 Stuck? Uncomment for a nudge
# hint("p07_autograd")

---
## 🔓 What this unlocks

With **Autograd Essentials** in your toolkit, you're ready for these problems:

| # | Problem | Difficulty | What you'll reuse | Links |
|:---:|---|:---:|---|---|
| #7 | BatchNorm | 🟡 Medium | Updating running stats from `batch_mean.detach()` so they never enter the graph | [Open](07_batchnorm.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/07_batchnorm.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/07_batchnorm.ipynb" target="_blank">Colab</a> |
| #20 | Kaiming Init | 🟢 Easy | Filling a weight in place under `torch.no_grad()` | [Open](20_weight_init.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/20_weight_init.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/20_weight_init.ipynb" target="_blank">Colab</a> |
| #42 | BatchNorm2d | 🟡 Medium | Updating per-channel running stats in place without entering the graph | [Open](42_batchnorm2d.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/42_batchnorm2d.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/42_batchnorm2d.ipynb" target="_blank">Colab</a> |
| #21 | Gradient Clipping | 🟢 Easy | Reading and rescaling `p.grad` for every parameter | [Open](21_gradient_clipping.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/21_gradient_clipping.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/21_gradient_clipping.ipynb" target="_blank">Colab</a> |
| #29 | Adam Optimizer | 🟡 Medium | `sgd_step_`: the in-place update loop Adam builds on | [Open](29_adam.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/29_adam.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/29_adam.ipynb" target="_blank">Colab</a> |
| #31 | Gradient Accumulation | 🟢 Easy | `accumulate_grads`: scale each micro-batch loss and let `.backward()` accumulate | [Open](31_gradient_accumulation.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/31_gradient_accumulation.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/31_gradient_accumulation.ipynb" target="_blank">Colab</a> |
| #37 | DPO Loss | 🔴 Hard | `stop_gradient_mse`-style constants: the reference model's log-probs | [Open](37_dpo_loss.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/37_dpo_loss.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/37_dpo_loss.ipynb" target="_blank">Colab</a> |
| #38 | GRPO Loss | 🔴 Hard | Detaching the advantages before weighting the log-probs | [Open](38_grpo_loss.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/38_grpo_loss.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/38_grpo_loss.ipynb" target="_blank">Colab</a> |
| #39 | PPO Loss | 🔴 Hard | Detaching the old log-probs and the advantages in the ratio | [Open](39_ppo_loss.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/39_ppo_loss.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/39_ppo_loss.ipynb" target="_blank">Colab</a> |
| #40 | Linear Regression | 🟡 Medium | `grad_of` to check the hand-derived gradient of the gradient-descent method | [Open](40_linear_regression.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/40_linear_regression.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/40_linear_regression.ipynb" target="_blank">Colab</a> |
| #41 | OPD Loss | 🔴 Hard | Detaching the teacher's logits | [Open](41_opd_loss.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/41_opd_loss.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/41_opd_loss.ipynb" target="_blank">Colab</a> |

🗺️ See the full recommended order in <a href="https://github.com/lalo/TorchCode/blob/master/LEARNING_PATH.md" target="_blank">LEARNING_PATH.md</a>.